# Tutorial: AlphaOmega Reporter Synthetic Demo

Audience:
- Users validating the report pipeline before running private AlphaOmega cases.

Prerequisites:
- Install the package in editable mode.
- Run this notebook from the repository root.

Learning goals:
- Generate a synthetic bilateral STN session.
- Build a beta-band PDF report.
- Render a multi-band report with placeholder panels for missing data.


## Outline

1. Setup
2. Create a synthetic session
3. Build a default beta-band report
4. Build a multi-band report with missing-data placeholders
5. Adapt the same flow to a real case directory


In [ ]:
from __future__ import annotations

from pathlib import Path

from pypdf import PdfReader

from alphaomega_reporter import ReportConfig
from alphaomega_reporter.report import build_report_from_session
from alphaomega_reporter.synthetic import make_synthetic_session

OUTPUT_DIR = Path("output/jupyter-notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

## Step 1 - Create a synthetic bilateral STN session

The synthetic generator produces depth-varying MER spiking and an LFP band peak near target depth 0 mm.


In [ ]:
session = make_synthetic_session(case_name="demo_case", band_peak="beta", random_seed=7)
summary = {
    "case_name": session.meta["case_name"],
    "n_segments": len(session.segments),
    "first_segment": session.segments[0].segment_id,
    "depths": sorted({seg.meta["depth_mm"] for seg in session.segments})[:4],
}
summary

## Step 2 - Build a default beta-band PDF report

This uses the same report builder as the CLI, but operates directly on the in-memory synthetic session.


In [ ]:
config = ReportConfig()
beta_report = OUTPUT_DIR / "synthetic_report_beta.pdf"
build_report_from_session(session, beta_report, config)
{"report": str(beta_report), "pages": len(PdfReader(str(beta_report)).pages)}

## Step 3 - Render multiple bands and missing-data placeholders

Here the synthetic LFP peak is shifted toward gamma, and one depth is missing LFP while another is missing spikes. The report should still build successfully.


In [ ]:
session_multiband = make_synthetic_session(
    case_name="demo_case_multiband",
    band_peak="gamma",
    random_seed=11,
    missing_lfp_depths={0.0},
    missing_spike_depths={-1.0},
)
config_multiband = ReportConfig()
config_multiband.bands.default_primary = "highbeta"
config_multiband.render.plot_bands = ["delta", "theta", "alpha", "beta", "gamma"]
multiband_report = OUTPUT_DIR / "synthetic_report_multiband.pdf"
build_report_from_session(session_multiband, multiband_report, config_multiband)
{"report": str(multiband_report), "pages": len(PdfReader(str(multiband_report)).pages)}

## Step 4 - Adapt to a real case directory

For real AlphaOmega cases, switch to the CLI:

```bash
ao-report build --case-dir /path/to/case_dir --out report.pdf
ao-report build --case-dir /path/to/case_dir --out report.pdf --band highbeta --plot-band gamma
ao-report export-h5 --case-dir /path/to/case_dir --out session.h5
```

Use only local private data for validation. Do not commit patient directories into the repository.
